# PETRUG: AI-Based Predictive Early Warning System for Pipeline Integrity Risk on AISI 4145H Steel Using Pressure and Flow Dynamics in Upstream Oil Operations

Notebook ini disusun mengikuti alur **CRISP-DM** untuk kebutuhan Proof of Concept. Fokus model adalah membaca pola tekanan dan dinamika aliran pada data SCADA sintetis, lalu menghasilkan status risiko operasional pipa: **Normal**, **Warning**, dan **Critical**.

---

## Business Understanding

**Konteks Permasalahan**  
Pipeline pada operasi hulu minyak bekerja pada kondisi tekanan dan aliran yang berubah-ubah. Keterlambatan dalam mengenali perubahan pola operasional dapat meningkatkan risiko gangguan pipeline, kehilangan produksi, serta kebutuhan maintenance yang lebih reaktif. Pada tahap Proof of Concept ini, PETRUG difokuskan sebagai sistem peringatan dini berbasis data untuk membantu membaca potensi risiko integritas pipa sejak awal.

**Tujuan Bisnis**  
Tujuan utama proyek ini adalah membangun model prediktif yang mampu mengklasifikasikan kondisi pipeline menjadi **Normal**, **Warning**, atau **Critical** berdasarkan data tekanan, laju aliran, kecepatan aliran, suhu, dan parameter operasional pendukung. Output model ditujukan sebagai alat bantu monitoring, bukan sebagai sistem kontrol otomatis terhadap peralatan lapangan.

**Objek Studi**  
Objek kajian difokuskan pada pipa berbahan **AISI 4145H Steel** sebagai representasi material baja paduan berkekuatan tinggi yang relevan untuk lingkungan operasi hulu minyak. Aspek material tidak dijadikan target prediksi utama, tetapi menjadi konteks engineering dalam menentukan batas operasional dan relevansi risiko.

**Indikator Keberhasilan**  
Indikator keberhasilan PoC meliputi kemampuan model dalam membedakan status risiko pipeline, performa evaluasi model menggunakan **accuracy**, **precision**, **recall**, **F1-score**, dan **confusion matrix**, serta kemampuan notebook untuk melakukan simulasi inference saat data baru masuk.

---

## Data Understanding

Dataset yang digunakan merupakan **data sintetis berbasis referensi parameter SCADA** untuk tahap Proof of Concept. Data ini digunakan untuk mensimulasikan kondisi operasional pipeline, termasuk tekanan, suhu, laju aliran, kecepatan aliran, perubahan tekanan, perubahan suhu, serta status risiko operasional. Penggunaan data sintetis dipilih karena data industri aktual umumnya bersifat sensitif dan belum tersedia pada tahap awal pengembangan.

In [1]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from statsmodels.tsa.seasonal import seasonal_decompose

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

### Load Data

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving PETRUG_AISI4145H_SCADA_Dataset.csv to PETRUG_AISI4145H_SCADA_Dataset.csv


In [ ]:
df = pd.read_csv("PETRUG_AISI4145H_SCADA_Dataset.csv")

In [ ]:
df.shape

(1200, 16)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Timestamp              1200 non-null   object 
 1   Segment_ID             1200 non-null   object 
 2   Pressure_MPa           1200 non-null   float64
 3   Temperature_C          1200 non-null   float64
 4   Flow_Rate_m3h          1200 non-null   float64
 5   Flow_Velocity_ms       1200 non-null   float64
 6   Strain_ustrain         1200 non-null   float64
 7   Acoustic_Emission_dB   1200 non-null   float64
 8   Flow_Direction         1200 non-null   object 
 9   dP_dt_MPa_min          1200 non-null   float64
 10  dT_dt_C_min            1200 non-null   float64
 11  Pipe_Integrity_Index   1200 non-null   float64
 12  RUL_days               1200 non-null   int64  
 13  Threat_Label           1200 non-null   object 
 14  Leak_Detection_Status  1200 non-null   object 
 15  Risk

In [ ]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Pressure_MPa,1200.0,8.235484,2.854241,4.2660,6.10525,7.79850,9.242000,17.3690
Temperature_C,1200.0,91.474200,26.603742,48.1800,71.63500,89.85500,106.825000,175.3800
Flow_Rate_m3h,1200.0,347.588333,35.439874,268.5000,321.40000,343.30000,366.700000,460.8000
Flow_Velocity_ms,1200.0,2.359518,0.721717,0.3510,1.87175,2.30850,2.736250,4.6580
Strain_ustrain,1200.0,0.048911,0.039484,0.0050,0.02120,0.03325,0.069500,0.2106
Acoustic_Emission_dB,1200.0,65.440833,9.173941,49.0000,59.20000,62.85000,69.800000,103.1000
dP_dt_MPa_min,1200.0,0.080805,0.104507,-0.1478,-0.00285,0.05015,0.178625,0.3614
dT_dt_C_min,1200.0,0.095534,0.332661,-0.8787,-0.12455,0.08655,0.300875,1.3111
Pipe_Integrity_Index,1200.0,81.869917,15.091585,27.5000,72.47500,87.75000,93.000000,100.0000
RUL_days,1200.0,2944.345000,1208.283669,51.0000,1957.75000,3681.00000,3838.000000,4123.0000


### Validasi Awal Dataset

In [ ]:
data_quality = pd.DataFrame({
    "missing_value": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "dtype": df.dtypes.astype(str)
})
data_quality

,missing_value,missing_percent,dtype
Timestamp,0,0.0,object
Segment_ID,0,0.0,object
Pressure_MPa,0,0.0,float64
Temperature_C,0,0.0,float64
Flow_Rate_m3h,0,0.0,float64
Flow_Velocity_ms,0,0.0,float64
Strain_ustrain,0,0.0,float64
Acoustic_Emission_dB,0,0.0,float64
Flow_Direction,0,0.0,object
dP_dt_MPa_min,0,0.0,float64


In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Jumlah data duplikat: {duplicate_count}")

Jumlah data duplikat: 0


### Distribusi Status Risiko

In [ ]:
df["Operational_Status"] = df["Leak_Detection_Status"].replace({"Leak_Detected": "Critical"})
status_order = ["Normal", "Warning", "Critical"]
status_counts = df["Operational_Status"].value_counts().reindex(status_order)
status_counts

,count
Operational_Status,
Normal,758
Warning,262
Critical,180


In [ ]:
fig = px.bar(
    status_counts.reset_index(),
    x="Operational_Status",
    y="count",
    text="count",
    title="Distribusi Status Risiko Pipeline",
    labels={"Operational_Status": "Status Risiko", "count": "Jumlah Data"},
)
fig.update_traces(textposition="outside")
fig.show()

### Exploratory Data Analysis
Visualisasi berikut membantu melihat distribusi parameter utama yang digunakan dalam model.

In [ ]:
main_features = ["Pressure_MPa", "Temperature_C", "Flow_Rate_m3h", "Flow_Velocity_ms"]
for col in main_features:
    fig = px.histogram(
        df,
        x=col,
        color="Operational_Status",
        nbins=40,
        marginal="box",
        title=f"Distribusi {col} berdasarkan Status Risiko",
        opacity=0.75,
    )
    fig.show()

### Pola Time Series Tekanan dan Aliran
Karena data bersifat time-based, grafik berikut digunakan untuk melihat perubahan tekanan dan aliran sepanjang waktu.

In [ ]:
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df_sorted = df.sort_values("Timestamp")

sample_ts = df_sorted.head(300)
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=sample_ts["Timestamp"], y=sample_ts["Pressure_MPa"], mode="lines", name="Pressure MPa"), secondary_y=False)
fig.add_trace(go.Scatter(x=sample_ts["Timestamp"], y=sample_ts["Flow_Rate_m3h"], mode="lines", name="Flow Rate m3h"), secondary_y=True)
fig.update_layout(title="Contoh Pola Time Series Tekanan dan Flow Rate", xaxis_title="Timestamp")
fig.update_yaxes(title_text="Pressure MPa", secondary_y=False)
fig.update_yaxes(title_text="Flow Rate m3h", secondary_y=True)
fig.show()

### Korelasi Antar Parameter Operasional

In [ ]:
corr_cols = [
    "Pressure_MPa", "Temperature_C", "Flow_Rate_m3h", "Flow_Velocity_ms",
    "Strain_ustrain", "Acoustic_Emission_dB", "dP_dt_MPa_min", "dT_dt_C_min"
]
corr = df[corr_cols].corr()
fig = px.imshow(
    corr,
    text_auto=True,
    aspect="auto",
    title="Heatmap Korelasi Parameter Operasional",
    color_continuous_scale="RdBu_r",
)
fig.show()

### Analisis Time Series dengan statsmodels
Bagian ini digunakan untuk membaca kecenderungan sederhana pada data tekanan. Pada tahap PoC, analisis ini tidak digunakan sebagai model utama, tetapi membantu memahami pola trend dan noise pada parameter operasional.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose

df["Timestamp"] = pd.to_datetime(df["Timestamp"])

ts_pressure = (
    df.sort_values("Timestamp")
      .set_index("Timestamp")["Pressure_MPa"]
      .resample("min")
      .mean()
)

ts_pressure = (
    ts_pressure
    .interpolate(method="time")
    .ffill()
    .bfill()
)

ts_sample = ts_pressure.head(720).dropna()

print("Jumlah data:", len(ts_sample))
print("Jumlah missing:", ts_sample.isna().sum())


if len(ts_sample) < 120:
    raise ValueError("Data belum cukup untuk seasonal decomposition dengan period=60.")

decomp = seasonal_decompose(
    ts_sample,
    model="additive",
    period=60,
    extrapolate_trend="freq"
)

decomp_df = pd.DataFrame({
    "Timestamp": ts_sample.index,
    "Observed": decomp.observed,
    "Trend": decomp.trend,
    "Seasonal": decomp.seasonal,
    "Residual": decomp.resid,
}).reset_index(drop=True)

fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    subplot_titles=["Observed", "Trend", "Seasonal", "Residual"]
)

for i, col in enumerate(["Observed", "Trend", "Seasonal", "Residual"], start=1):
    fig.add_trace(
        go.Scatter(
            x=decomp_df["Timestamp"],
            y=decomp_df[col],
            mode="lines",
            name=col
        ),
        row=i,
        col=1
    )

fig.update_layout(
    height=800,
    title="Dekomposisi Time Series Pressure MPa dengan statsmodels",
    showlegend=False
)

fig.show()

Jumlah data: 720
Jumlah missing: 0


---

## Data Preparation

Tahap ini menyiapkan data agar layak digunakan untuk pelatihan model. Label operasional dibuat menjadi tiga kelas utama: **Normal**, **Warning**, dan **Critical**. Beberapa fitur turunan juga dibuat dari tekanan dan dinamika aliran untuk membantu model membaca pola risiko.

In [ ]:
model_df = df.copy()

model_df["Operational_Status"] = model_df["Leak_Detection_Status"].replace({
    "Leak_Detected": "Critical"
})

model_df["Pressure_Flow_Ratio"] = model_df["Pressure_MPa"] / (model_df["Flow_Rate_m3h"] + 1e-6)
model_df["Velocity_Flow_Ratio"] = model_df["Flow_Velocity_ms"] / (model_df["Flow_Rate_m3h"] + 1e-6)
model_df["Acoustic_Strain_Index"] = model_df["Acoustic_Emission_dB"] * model_df["Strain_ustrain"]
model_df["Temperature_Pressure_Index"] = model_df["Temperature_C"] * model_df["Pressure_MPa"]
model_df["Pressure_Change_Abs"] = model_df["dP_dt_MPa_min"].abs()
model_df["Temperature_Change_Abs"] = model_df["dT_dt_C_min"].abs()

model_df[
    [
        "Pressure_Flow_Ratio",
        "Velocity_Flow_Ratio",
        "Acoustic_Strain_Index",
        "Temperature_Pressure_Index",
        "Pressure_Change_Abs",
        "Temperature_Change_Abs"
    ]
].head()

,Pressure_Flow_Ratio,Velocity_Flow_Ratio,Acoustic_Strain_Index,Temperature_Pressure_Index,Pressure_Change_Abs,Temperature_Change_Abs
0,0.023271,0.004325,0.84189,471.91080,0.0078,0.6457
1,0.025657,0.006841,1.74580,525.97620,0.0057,0.0382
2,0.018873,0.007358,0.53932,376.19792,0.0184,0.3108
3,0.017002,0.006432,1.08360,405.31796,0.0697,0.0445
4,0.023228,0.004514,7.87284,929.61414,0.2979,0.1253


### Pemilihan Fitur dan Target


In [ ]:
target_col = "Operational_Status"

feature_cols = [
    "Pressure_MPa",
    "Flow_Rate_m3h",
    "Flow_Velocity_ms",
    "Temperature_C",
    "dP_dt_MPa_min",
    "dT_dt_C_min",
    "Pressure_Flow_Ratio",
    "Temperature_Pressure_Index"
]
X = model_df[feature_cols].copy()
y = model_df[target_col].copy()

categorical_features = ["Flow_Direction"]
numeric_features = [col for col in feature_cols if col not in categorical_features]

print("Fitur final yang digunakan:")
print(feature_cols)

print("Jumlah fitur:", len(feature_cols))
print("Fitur numerik:", numeric_features)
print("Fitur kategorikal:", categorical_features)

print("\nDistribusi target:")
print(y.value_counts())

Fitur final yang digunakan:
['Pressure_MPa', 'Flow_Rate_m3h', 'Flow_Velocity_ms', 'Temperature_C', 'dP_dt_MPa_min', 'dT_dt_C_min', 'Pressure_Flow_Ratio', 'Temperature_Pressure_Index']
Jumlah fitur: 8
Fitur numerik: ['Pressure_MPa', 'Flow_Rate_m3h', 'Flow_Velocity_ms', 'Temperature_C', 'dP_dt_MPa_min', 'dT_dt_C_min', 'Pressure_Flow_Ratio', 'Temperature_Pressure_Index']
Fitur kategorikal: ['Flow_Direction']

Distribusi target:
Operational_Status
Normal      758
Warning     262
Critical    180
Name: count, dtype: int64


### Data Splitting
Data dibagi menjadi data latih dan data uji dengan stratifikasi agar proporsi kelas tetap terjaga. Pembagian ini penting untuk memastikan evaluasi model tidak hanya baik pada data pelatihan.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Distribusi target data latih:")
print(y_train.value_counts(normalize=True).round(3))
print("Distribusi target data uji:")
print(y_test.value_counts(normalize=True).round(3))

Train shape: (960, 8)
Test shape: (240, 8)
Distribusi target data latih:
Operational_Status
Normal      0.631
Warning     0.219
Critical    0.150
Name: proportion, dtype: float64
Distribusi target data uji:
Operational_Status
Normal      0.633
Warning     0.217
Critical    0.150
Name: proportion, dtype: float64


### Preprocessing Pipeline
Pipeline digunakan agar proses imputasi, scaling, encoding, dan pelatihan model berjalan konsisten. Ini penting agar model yang disimpan nanti dapat digunakan kembali pada tahap inference.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
    ]
)

---

## Modeling

Pada tahap modeling, beberapa algoritma dibandingkan untuk mencari model yang paling stabil. Model utama yang diprioritaskan adalah **Gradient Boosting**, sedangkan **Random Forest** digunakan sebagai pembanding kuat, dan **Logistic Regression** digunakan sebagai baseline. Tambahan SVM, KNN, dan Decision Tree disertakan untuk memperkaya perbandingan model.

In [ ]:
models = {
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        random_state=RANDOM_STATE,
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        class_weight="balanced",
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        C=0.5,
        penalty="l2",
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),

    "Support Vector Machine": SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        probability=True,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),

    "K-Nearest Neighbour": KNeighborsClassifier(
        n_neighbors=15,
        weights="distance"
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_split=15,
        min_samples_leaf=7,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    ),
}

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline

selected_model = models["Gradient Boosting"]

pipeline_for_cv = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", selected_model),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    pipeline_for_cv,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("CV Scores:", cv_scores)
print("Mean Accuracy:", cv_scores.mean())
print("Std Accuracy:", cv_scores.std())

CV Scores: [0.9875     0.97083333 0.96666667 0.99166667 0.9875    ]
Mean Accuracy: 0.9808333333333333
Std Accuracy: 0.010069204977995495


### Training dan Perbandingan Model

In [ ]:
trained_models = {}
model_results = []
class_order = ["Normal", "Warning", "Critical"]

for name, estimator in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator),
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    trained_models[name] = pipeline
    model_results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_weighted": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    })

results_df = pd.DataFrame(model_results).sort_values("f1_weighted", ascending=False).reset_index(drop=True)
results_df

,model,accuracy,precision_weighted,recall_weighted,f1_weighted
0,Random Forest,0.983333,0.983741,0.983333,0.983435
1,Support Vector Machine,0.979167,0.979414,0.979167,0.979158
2,Gradient Boosting,0.975000,0.975506,0.975000,0.975153
3,K-Nearest Neighbour,0.966667,0.966470,0.966667,0.966180
4,Decision Tree,0.962500,0.968033,0.962500,0.963421
5,Logistic Regression,0.912500,0.915766,0.912500,0.913726


In [ ]:
fig = px.bar(
    results_df,
    x="model",
    y=["accuracy", "precision_weighted", "recall_weighted", "f1_weighted"],
    barmode="group",
    title="Perbandingan Performa Model",
    labels={"value": "Score", "model": "Model", "variable": "Metric"},
)
fig.update_yaxes(range=[0, 1.05])
fig.show()

### Model Terbaik

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = trained_models[best_model_name]
print("Model terbaik:", best_model_name)
print("Accuracy:", round(results_df.iloc[0]["accuracy"], 4))
print("F1 weighted:", round(results_df.iloc[0]["f1_weighted"], 4))

Model terbaik: Random Forest
Accuracy: 0.9833
F1 weighted: 0.9834


---

## Evaluation

Evaluasi dilakukan menggunakan confusion matrix, accuracy, precision, recall, dan F1-score. Pada kasus pipeline integrity, recall untuk kelas risiko tinggi juga penting karena sistem peringatan dini perlu mengurangi kemungkinan kondisi kritis yang terlewat.

In [ ]:
y_pred_best = best_model.predict(X_test)
print(classification_report(y_test, y_pred_best, labels=class_order, zero_division=0))

              precision    recall  f1-score   support

      Normal       0.99      0.98      0.99       152
     Warning       0.94      0.98      0.96        52
    Critical       1.00      1.00      1.00        36

    accuracy                           0.98       240
   macro avg       0.98      0.99      0.98       240
weighted avg       0.98      0.98      0.98       240



### Confusion Matrix Model Terbaik

In [ ]:
cm = confusion_matrix(y_test, y_pred_best, labels=class_order)
cm_df = pd.DataFrame(cm, index=class_order, columns=class_order)

fig = px.imshow(
    cm_df,
    text_auto=True,
    color_continuous_scale="Blues",
    title=f"Confusion Matrix - {best_model_name}",
    labels={"x": "Predicted", "y": "Actual", "color": "Count"},
)
fig.show()
cm_df

,Normal,Warning,Critical
Normal,149,3,0
Warning,1,51,0
Critical,0,0,36


### Evaluasi Semua Model dengan Confusion Matrix

In [ ]:
for name, model in trained_models.items():
    pred = model.predict(X_test)
    cm = confusion_matrix(y_test, pred, labels=class_order)
    cm_df = pd.DataFrame(cm, index=class_order, columns=class_order)
    fig = px.imshow(
        cm_df,
        text_auto=True,
        color_continuous_scale="Blues",
        title=f"Confusion Matrix - {name}",
        labels={"x": "Predicted", "y": "Actual", "color": "Count"},
    )
    fig.show()

### Interpretasi Evaluasi
Jika hasil model menunjukkan akurasi tinggi pada data PoC, interpretasinya tetap harus ditempatkan secara tepat. Dataset yang digunakan adalah data sintetis berbasis pola SCADA, sehingga hasil evaluasi menunjukkan bahwa alur model dan fitur bekerja baik pada skenario PoC. Pada tahap berikutnya, model perlu divalidasi ulang menggunakan data historis operasional yang lebih mendekati kondisi lapangan.

### Simpan Model dan Artefak
Bagian ini penting agar notebook tidak berhenti di eksperimen saja. Model terbaik, daftar fitur, hasil evaluasi, dan metadata disimpan sebagai artefak yang bisa digunakan kembali untuk dashboard atau inference.

In [ ]:
joblib.dump(best_model,"petrugmodel.pkl")
results_df.to_csv("datasetkedua.csv", index=False)

metadata = {
    "project_title": "PETRUG: AI-Based Predictive Early Warning System for Pipeline Integrity Risk on AISI 4145H Steel",
    "best_model": best_model_name,
    "target": target_col,
    "feature_columns": feature_cols,
    "class_order": class_order,
    "test_size": 0.2,
    "random_state": RANDOM_STATE,
    "notes": "PoC menggunakan synthetic SCADA-like dataset. Kolom target-derived tidak digunakan sebagai input model.",
}

with open("model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Model dan artefak berhasil disimpan di folder:", MODELS_DIR.resolve())

Model dan artefak berhasil disimpan di folder: /content/models


---

## Deployment

Pada CRISP-DM, deployment pada tahap PoC tidak harus berarti implementasi penuh di industri. Untuk notebook ini, deployment ditunjukkan melalui simulasi inference: data baru masuk, fitur turunan dihitung, lalu model memberikan prediksi status risiko secara langsung.

### Fungsi Preprocessing untuk Data Baru

In [ ]:
def prepare_single_input(input_data: dict) -> pd.DataFrame:
    """Menyiapkan satu data baru agar formatnya sama dengan data training."""
    row = pd.DataFrame([input_data])
    row["Pressure_Flow_Ratio"] = row["Pressure_MPa"] / (row["Flow_Rate_m3h"] + 1e-6)
    row["Velocity_Flow_Ratio"] = row["Flow_Velocity_ms"] / (row["Flow_Rate_m3h"] + 1e-6)
    row["Acoustic_Strain_Index"] = row["Acoustic_Emission_dB"] * row["Strain_ustrain"]
    row["Temperature_Pressure_Index"] = row["Temperature_C"] * row["Pressure_MPa"]
    row["Pressure_Change_Abs"] = row["dP_dt_MPa_min"].abs()
    row["Temperature_Change_Abs"] = row["dT_dt_C_min"].abs()
    return row[feature_cols]


def predict_pipeline_status(input_data: dict, model=best_model) -> pd.DataFrame:
    """Menghasilkan prediksi status risiko dan probabilitas kelas jika tersedia."""
    prepared = prepare_single_input(input_data)
    prediction = model.predict(prepared)[0]
    output = {"Predicted_Status": prediction}

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(prepared)[0]
        classes = model.classes_
        for cls, prob in zip(classes, proba):
            output[f"Probability_{cls}"] = round(float(prob), 4)

    return pd.DataFrame([output])

### Simulasi Inference Data Masuk
Contoh berikut mensimulasikan satu data operasional baru. Ketika data masuk, sistem langsung menghasilkan prediksi status pipeline.

In [ ]:
new_sample_normal = {
    "Segment_ID": "SEG-A1",
    "Pressure_MPa": 8.2,
    "Temperature_C": 58.0,
    "Flow_Rate_m3h": 350.0,
    "Flow_Velocity_ms": 2.3,
    "Strain_ustrain": 0.02,
    "Acoustic_Emission_dB": 60.0,
    "Flow_Direction": "Forward",
    "dP_dt_MPa_min": 0.01,
    "dT_dt_C_min": 0.05,
}

predict_pipeline_status(new_sample_normal)

,Predicted_Status,Probability_Critical,Probability_Normal,Probability_Warning
0,Normal,0.0009,0.9758,0.0234


In [ ]:
new_sample_risk = {
    "Segment_ID": "SEG-B1",
    "Pressure_MPa": 15.8,
    "Temperature_C": 145.0,
    "Flow_Rate_m3h": 285.0,
    "Flow_Velocity_ms": 0.7,
    "Strain_ustrain": 0.18,
    "Acoustic_Emission_dB": 96.0,
    "Flow_Direction": "Reverse",
    "dP_dt_MPa_min": 0.30,
    "dT_dt_C_min": 1.05,
}

predict_pipeline_status(new_sample_risk)

,Predicted_Status,Probability_Critical,Probability_Normal,Probability_Warning
0,Critical,0.9598,0.0001,0.0401


### Simulasi Streaming Sederhana
Bagian ini mengambil beberapa baris terakhir dari data uji dan memprediksi statusnya satu per satu. Ini menggambarkan alur sederhana saat data baru masuk secara berurutan.

In [ ]:
stream_sample = X_test.head(10).copy()
stream_pred = best_model.predict(stream_sample)
stream_result = stream_sample.copy()
stream_result["Predicted_Status"] = stream_pred
stream_result[[ "Pressure_MPa", "Flow_Rate_m3h", "Flow_Velocity_ms", "Predicted_Status"]]

,Pressure_MPa,Flow_Rate_m3h,Flow_Velocity_ms,Predicted_Status
589,5.232,366.7,1.938,Normal
780,8.899,358.9,2.361,Normal
521,8.116,352.1,1.209,Critical
858,12.624,300.2,1.137,Critical
1071,7.827,324.3,2.205,Normal
1191,5.240,356.4,1.638,Normal
114,8.603,291.2,1.123,Critical
1180,9.533,293.3,2.320,Warning
683,8.254,348.6,1.963,Normal
1019,6.762,355.2,2.835,Normal


### Kesimpulan
Berdasarkan tahapan CRISP-DM yang dilakukan, PETRUG menunjukkan bahwa data tekanan dan dinamika aliran dapat digunakan untuk membangun sistem prediksi risiko pipeline pada tahap Proof of Concept. Model terbaik dipilih berdasarkan evaluasi perbandingan beberapa algoritma dan sudah dapat digunakan untuk simulasi inference ketika data baru masuk. Hasil ini menjadi dasar awal untuk pengembangan dashboard dan validasi lanjutan menggunakan data operasional yang lebih dekat dengan kondisi lapangan.